# Session 1 — Data preparation and spatial context

Load and validate the paired Tonsil data, apply the official DGAT quality-control and normalization workflow, and construct spatial context.

For workshop reproducibility, select **Runtime → Change runtime type → 2026.04**
(Python 3.12) before running the bootstrap.

Work through the parts in order. Outputs and JSON checkpoints are written to
`MyDrive/ECCB2026/state`, so they survive a Colab runtime reset. If the runtime stops,
rerun the bootstrap cell, inspect the printed completed checkpoints, and jump to the
first unfinished part; each part reloads its required inputs.


In [ ]:
from pathlib import Path
import importlib
import importlib.util
import json
import os
import shutil
import subprocess
import sys

SESSION_REQUIREMENTS = [('anndata', 'anndata==0.11.4'), ('scanpy', 'scanpy==1.11.5'), ('muon', 'muon==0.1.7')]
NEED_DGAT = True

in_colab = importlib.util.find_spec("google.colab") is not None and Path("/content").is_dir()
if in_colab:
    from google.colab import drive

    drive.mount("/content/drive", force_remount=False)
    repo_dir = Path("/content/ECCB-2026-Tutorial")
    tutorial_root = repo_dir / "hands-on_tutorial"
    if not (tutorial_root / "src" / "dgat_tutorial").is_dir():
        subprocess.run(
            ["git", "clone", "--depth", "1",
             "https://github.com/osmanbeyoglulab/ECCB-2026-Tutorial.git", str(repo_dir)],
            check=True,
        )
    else:
        # A Colab runtime can outlive the notebook tab. Refresh an existing clone
        # so newly opened notebooks do not silently execute stale setup scripts.
        subprocess.run(
            ["git", "-C", str(repo_dir), "fetch", "--depth", "1", "origin", "main"],
            check=True,
        )
        subprocess.run(
            ["git", "-C", str(repo_dir), "reset", "--hard", "origin/main"],
            check=True,
        )

    drive_root = Path("/content/drive/MyDrive/ECCB2026")
    drive_data = drive_root / "assets" / "DGAT_assets" / "data"
    manifest_path = drive_root / "asset_manifest.json"
    if not manifest_path.is_file():
        raise FileNotFoundError(f"Missing {manifest_path}. Run Session 0 before the tutorial.")
    asset_manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
    local_data = tutorial_root / "external" / "DGAT_assets" / "data"
    local_data.mkdir(parents=True, exist_ok=True)
    for filename in ("Tonsil_RNA.h5ad", "Tonsil_ADT.h5ad"):
        source = drive_data / filename
        destination = local_data / filename
        if not source.is_file() or source.stat().st_size == 0:
            raise FileNotFoundError(
                f"Missing {source}. Run Session 0 Drive preparation before the tutorial."
            )
        expected_bytes = asset_manifest["files"][filename]["bytes"]
        if source.stat().st_size != expected_bytes:
            raise IOError(
                f"Drive asset size mismatch for {filename}: expected {expected_bytes}, "
                f"found {source.stat().st_size}. Rerun Session 0."
            )
        if not destination.is_file() or destination.stat().st_size != source.stat().st_size:
            print(f"Copying {filename} from Drive to the Colab VM ...")
            shutil.copy2(source, destination)

    os.environ["DGAT_TUTORIAL_STATE_DIR"] = str(drive_root / "state")

    if NEED_DGAT:
        dgat_dir = tutorial_root / "external" / "DGAT"
        if not (dgat_dir / "utils" / "Preprocessing.py").is_file():
            dgat_dir.parent.mkdir(parents=True, exist_ok=True)
            subprocess.run(
                ["git", "clone", "--depth", "1",
                 "https://github.com/osmanbeyoglulab/DGAT.git", str(dgat_dir)],
                check=True,
            )

    missing_specs = [
        package_spec
        for import_name, package_spec in SESSION_REQUIREMENTS
        if importlib.util.find_spec(import_name) is None
    ]
    if missing_specs:
        wheelhouse = drive_root / "wheelhouse" / f"py{sys.version_info.major}{sys.version_info.minor}"
        online_command = [sys.executable, "-m", "pip", "install", "-q", *missing_specs]
        if wheelhouse.is_dir() and any(wheelhouse.glob("*.whl")):
            print(f"Installing missing packages using the Drive wheel cache: {missing_specs}")
            cached_command = [
                sys.executable, "-m", "pip", "install", "-q", "--no-index",
                "--find-links", str(wheelhouse), *missing_specs,
            ]
            try:
                subprocess.run(cached_command, check=True)
            except subprocess.CalledProcessError:
                print("Wheel cache was incomplete; falling back to PyPI.")
                subprocess.run(online_command, check=True)
        else:
            print(f"Drive wheel cache missing; installing from PyPI: {missing_specs}")
            subprocess.run(online_command, check=True)
        importlib.invalidate_caches()
else:
    candidates = [Path.cwd().resolve(), *Path.cwd().resolve().parents]
    for candidate in candidates:
        if (candidate / "src" / "dgat_tutorial").is_dir():
            tutorial_root = candidate
            break
    else:
        raise FileNotFoundError("Could not locate hands-on_tutorial/ from the current directory.")

os.chdir(tutorial_root)
src_dir = tutorial_root / "src"
if str(src_dir) not in sys.path:
    sys.path.insert(0, str(src_dir))

from dgat_tutorial.checkpoints import tutorial_paths, write_checkpoint

paths = tutorial_paths(tutorial_root)
completed = sorted(path.name for path in paths.checkpoints.glob("session_*/part_*.json"))
print(f"Tutorial root: {paths.root}")
print(f"Persistent state: {paths.checkpoints.parent}")
print("Completed checkpoints:", completed or "none yet")


## Session 1 · Part 1 — Build the tutorial data object

**Goal:** load paired spatial RNA and protein measurements into one validated object. We use the
lightweight `SpatialOmicsData` object. The
object plays the same teaching role: it keeps observations, modalities, and coordinates aligned.

By the end you should be able to explain why row identity and coordinate validation must happen
before normalization or graph construction. Every tutorial section uses the same **10x Genomics
CytAssist Tonsil** RNA/ADT pair downloaded during Session 0; there is no generated-data substitute.


#### Dataset — 10x Genomics CytAssist Human Tonsil

This tutorial uses a **public Visium CytAssist FFPE Human Tonsil** sample with paired gene expression
and protein (ADT) measurements (`CytAssist_FFPE_Protein_Expression_Human_Tonsil`).

| Item | Detail |
| --- | --- |
| Technology | 10x Genomics Visium CytAssist on FFPE tissue |
| Tissue | Human tonsil (immune secondary lymphoid organ with follicles / germinal centers) |
| Modalities | Spatial RNA + antibody-derived tags (proteins) on the same spots |
| Files | `Tonsil_RNA.h5ad`, `Tonsil_ADT.h5ad` (DGAT-packaged AnnData) |
| Approximate size | ~4,200 spots × ~18,000 genes × 35 proteins (before QC) |
| Why it is useful here | Paired RNA–protein labels let us validate imputed protein landscapes against measured ADT while still practicing an RNA→protein inference workflow |

Tonsil is a structured tissue: B- and T-cell zones create clear spatial protein patterns (for example
immune markers), which makes QC failures, graph neighborhoods, and prediction errors easier to interpret
visually. Downstream notebooks keep this same sample; there is no synthetic substitute.


### 1. Load all three linked tables


In [ ]:
import numpy as np
import pandas as pd

from dgat_tutorial.data import find_dgat_h5ad_pair, load_tutorial_data
from dgat_tutorial.processing import validate_modalities

pair = find_dgat_h5ad_pair(paths.raw_data)
dataset = load_tutorial_data(paths.raw_data)
spots = dataset.spots.copy()
# Keep all feature columns; validate_modalities will fail if non-numeric values appear.
transcripts = dataset.transcripts.copy()
proteins = dataset.proteins.copy()
dropped_rna = [c for c in transcripts.columns if not pd.api.types.is_numeric_dtype(transcripts[c])]
dropped_protein = [c for c in proteins.columns if not pd.api.types.is_numeric_dtype(proteins[c])]
if dropped_rna or dropped_protein:
    raise TypeError(
        "Non-numeric feature columns detected; refuse silent drop. "
        f"RNA={dropped_rna}; protein={dropped_protein}"
    )
source = f"RNA={pair[0]}, ADT={pair[1]}"

print(type(dataset).__name__)
print(f"Source: {source}")
print(f"spots × genes: {transcripts.shape}; spots × proteins: {proteins.shape}")
display(spots.head(3), transcripts.iloc[:3, :5], proteins.iloc[:3, :5])


### 2. Validate the object

DGAT assumes that row *i* in RNA, protein, and spatial coordinates is the same biological spot.
The next call checks ordered IDs, duplicate IDs, finite numeric x/y coordinates, finite values, and
non-negative abundance. A shape match alone is not sufficient.


In [ ]:
validate_modalities(spots, transcripts, proteins)
xy = spots[["x", "y"]].to_numpy(dtype=float)
checks = pd.DataFrame({
    "check": [
        "ordered IDs match",
        "x/y columns present",
        "x/y finite numeric",
        "finite RNA",
        "finite protein",
        "non-negative values",
    ],
    "passed": [
        spots.index.equals(transcripts.index) and spots.index.equals(proteins.index),
        {"x", "y"}.issubset(spots.columns),
        bool(np.isfinite(xy).all()),
        np.isfinite(transcripts.to_numpy(dtype=float)).all(),
        np.isfinite(proteins.to_numpy(dtype=float)).all(),
        (transcripts.to_numpy(dtype=float) >= 0).all() and (proteins.to_numpy(dtype=float) >= 0).all(),
    ],
})
checks


### 3. Record a reproducible input summary


In [ ]:
summary = pd.DataFrame([{
    "source": source,
    "spots": len(spots),
    "genes": transcripts.shape[1],
    "proteins": proteins.shape[1],
    "has_xy_coordinates": {"x", "y"}.issubset(spots.columns),
    "all_validation_checks_passed": bool(checks["passed"].all()),
}])
summary_path = paths.results / "session01_dataset_summary.csv"
summary.to_csv(summary_path, index=False)
manifest = write_checkpoint("1.1", [summary_path], summary=summary.iloc[0].to_dict(), start=paths.root)
summary


### Check

Continue only when every validation row is `True`. If IDs disagree, fix the upstream pairing—never
sort modalities independently and hope they align.


## Session 1 · Part 2 — QC, filtering, and normalization

**Goal:** apply the official **paired CITE-seq / training** preprocessing from
`external/DGAT/utils/Preprocessing.py` (`qc_control_cytassist` + `normalize`):

- keep spots with ≥700 detected genes
- remove spots with mitochondrial UMIs ≥35%
- keep genes detected in ≥2.5% of spots (plus encoding genes matched to the protein panel)
- remove ADT isotype controls (`mouse_`/`rat_` prefixes)
- RNA: library-size normalize to 10,000 → `log1p` → gene-wise scale with clip at 10
- protein: CLR via ``muon.prot.pp.clr`` (default ``axis=0``: per protein across spots)

**Important:** ST-only inference uses a lighter path (`preprocess_ST`: min_genes=700 +
RNA normalize/scale; no MT filter, no gene-prevalence filter, no protein CLR). Session 2's
official prediction wrapper follows that inference path. This notebook teaches the
*training* CytAssist recipe used when both RNA and protein are observed.


### 1. Calculate spot-level QC metrics


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from dgat_tutorial.data import load_tutorial_data
from dgat_tutorial.plotting import plot_qc_overview, plot_raw_vs_normalized
from dgat_tutorial.processing import (
    DGAT_MAX_MT_PCT,
    DGAT_MIN_CELLS_FRACTION,
    DGAT_MIN_GENES,
    calculate_qc_metrics,
    process_modalities_official_dgat,
    validate_modalities,
)

dataset = load_tutorial_data(paths.raw_data)
spots = dataset.spots.copy()
# Keep every gene/protein column; do not silently drop via select_dtypes.
transcripts = dataset.transcripts.apply(pd.to_numeric, errors="raise")
proteins = dataset.proteins.apply(pd.to_numeric, errors="raise")
validate_modalities(spots, transcripts, proteins)
qc_before = calculate_qc_metrics(transcripts, proteins)
qc_before.describe().T


#### Figure 1 — RNA and protein QC distributions


In [ ]:
qc_figure, _ = plot_qc_overview(qc_before)
qc_figure_path = paths.figures / "session01_qc_rna_and_protein.png"
qc_figure.savefig(qc_figure_path, dpi=160, bbox_inches="tight")
plt.show()


**How to read it:** low RNA totals and gene counts flag weak transcript capture; protein
totals reveal a different assay channel. Official DGAT training then applies fixed CytAssist
thresholds (700 genes, 35% MT, 2.5% gene prevalence) rather than sample-adaptive quantiles.


### 2. Apply official DGAT training filters


In [ ]:
processed = process_modalities_official_dgat(
    spots,
    transcripts,
    proteins,
    dgat_repo_dir=paths.root / "external" / "DGAT",
    min_genes=DGAT_MIN_GENES,
    max_mt_pct=DGAT_MAX_MT_PCT,
)
filtered_spots = processed.spots
filtered_rna = processed.raw_transcripts
filtered_protein = processed.raw_proteins
keep_mask = spots.index.astype(str).isin(filtered_spots.index)
filtering_summary = pd.DataFrame([{
    "min_genes": DGAT_MIN_GENES,
    "max_mt_pct": DGAT_MAX_MT_PCT,
    "min_gene_prevalence": DGAT_MIN_CELLS_FRACTION,
    "spots_before": len(spots), "spots_after": len(filtered_spots),
    "genes_before": transcripts.shape[1], "genes_after": filtered_rna.shape[1],
    "proteins_before": proteins.shape[1], "proteins_after": filtered_protein.shape[1],
}])
filtering_summary.T


#### Why do 35 protein features become 31?

The original ADT matrix contains 31 biological protein targets plus four antibody isotype
controls: `mouse_IgG2a`, `mouse_IgG1k`, `mouse_IgG2bk`, and `rat_IgG2a`. These controls help
characterize nonspecific antibody background; they are not protein targets predicted by DGAT.
The upstream call `qc_control_cytassist(..., remove_isotype=True)` removes features whose names
start with `mouse_` or `rat_`, leaving the 31-protein panel used downstream.


### 3. Official DGAT normalization


In [ ]:
# These matrices were produced above by DGAT's own normalize(...).
rna_normalized = processed.normalized_transcripts
protein_normalized = processed.normalized_proteins

normalization_checks = pd.DataFrame({
    "quantity": [
        "RNA gene-wise mean after scale",
        "mean CLR protein value per protein (muon axis=0)",
    ],
    "expected": ["≈0 after scale", "not necessarily 0 (axis=0 CLR)"],
    "observed_median": [
        float(rna_normalized.mean(axis=0).median()),
        float(protein_normalized.mean(axis=0).median()),
    ],
})
normalization_checks


#### Figure 2 — What normalization changes


In [ ]:
rna_feature = filtered_rna.var(axis=0).idxmax()
protein_feature = filtered_protein.var(axis=0).idxmax()
rna_fig, _ = plot_raw_vs_normalized(filtered_rna, rna_normalized, rna_feature, "RNA")
protein_fig, _ = plot_raw_vs_normalized(filtered_protein, protein_normalized, protein_feature, "protein")
rna_norm_path = paths.figures / "session01_rna_raw_vs_normalized.png"
protein_norm_path = paths.figures / "session01_protein_raw_vs_normalized.png"
rna_fig.savefig(rna_norm_path, dpi=160, bbox_inches="tight")
protein_fig.savefig(protein_norm_path, dpi=160, bbox_inches="tight")
plt.show()


### 4. Save processed matrices and a QC audit trail


In [ ]:
qc_path = paths.results / "session01_spot_qc.csv"
filtering_path = paths.results / "session01_filtering_summary.csv"
filtered_spots_path = paths.processed_data / "filtered_spots.csv"
filtered_rna_path = paths.processed_data / "filtered_rna_counts.csv"
filtered_protein_path = paths.processed_data / "filtered_protein_counts.csv"
rna_path = paths.processed_data / "rna_log_normalized.csv"
protein_path = paths.processed_data / "protein_clr_normalized.csv"
processed_manifest_path = paths.processed_data / "session01_preprocessing_outputs.csv"

qc_before.assign(kept=keep_mask).to_csv(qc_path)
filtering_summary.to_csv(filtering_path, index=False)
filtered_spots.to_csv(filtered_spots_path)
filtered_rna.to_csv(filtered_rna_path)
filtered_protein.to_csv(filtered_protein_path)
rna_normalized.to_csv(rna_path)
protein_normalized.to_csv(protein_path)

processed_outputs = pd.DataFrame([
    {
        "artifact": path.name,
        "stage": stage,
        "rows": table.shape[0],
        "columns": table.shape[1],
        "description": description,
    }
    for path, stage, table, description in [
        (filtered_spots_path, "filtered", filtered_spots, "Aligned spot metadata and x/y coordinates"),
        (filtered_rna_path, "filtered", filtered_rna, "RNA counts after spot and gene filtering"),
        (filtered_protein_path, "filtered", filtered_protein, "ADT counts after spot and isotype filtering"),
        (rna_path, "normalized", rna_normalized, "DGAT library-size/log1p/gene-scaled RNA"),
        (protein_path, "normalized", protein_normalized, "DGAT CLR-normalized protein values"),
    ]
])
processed_outputs.to_csv(processed_manifest_path, index=False)

processed_artifacts = [
    filtered_spots_path,
    filtered_rna_path,
    filtered_protein_path,
    rna_path,
    protein_path,
    processed_manifest_path,
]
manifest = write_checkpoint(
    "1.2",
    [qc_path, filtering_path, *processed_artifacts, qc_figure_path, rna_norm_path, protein_norm_path],
    summary={
        "spots_kept": len(filtered_spots),
        "spots_removed": int((~keep_mask).sum()),
        "processed_data_files": len(processed_artifacts),
    },
    start=paths.root,
)
print(f"Processed data saved under: {paths.processed_data}")
display(processed_outputs)
print(f"Checkpoint written: {manifest}")


### Check

You should now be able to justify: (1) which spots/features were removed, (2) why RNA and protein
use different transforms, and (3) why filtered counts remain available alongside normalized
matrices for QC and auditability. Confirm the six files listed above exist in `data/processed/`.


## Session 1 · Part 3 — Spatial neighborhoods and exploratory structure

**Goal:** construct the **official DGAT graphs** and inspect exploratory structure.

Official DGAT (`utils/Graph_utils.py`) builds:

1. a **spatial 6-NN** graph
2. an **RNA molecular 10-NN** graph in PCA space (when >1500 genes; 85% variance)
3. a **protein molecular 10-NN** graph in feature space

The RNA encoder uses `spatial ∪ RNA-molecular` edges; the protein encoder uses
`spatial ∪ protein-molecular` edges. This notebook visualizes the **spatial component
only** for readability, then saves the **union** graphs that the encoders actually use.


### 1. Reload and process the data independently


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA

from dgat_tutorial.data import load_tutorial_data
from dgat_tutorial.plotting import plot_spatial_feature, plot_spatial_knn_graph
from dgat_tutorial.processing import build_dgat_graphs, knn_edge_index, process_modalities_official_dgat

dataset = load_tutorial_data(paths.raw_data)
processed = process_modalities_official_dgat(
    dataset.spots,
    dataset.transcripts,
    dataset.proteins,
    dgat_repo_dir=paths.root / "external" / "DGAT",
)
spots = processed.spots
rna = processed.normalized_transcripts
protein = processed.normalized_proteins
graphs = build_dgat_graphs(spots, rna, protein)
edge_index = graphs["spatial_edge_index"]
print(
    f"Spatial 6-NN edges: {graphs['spatial_edge_index'].shape[1]}; "
    f"RNA union graph: {graphs['rna_edge_index'].shape[1]}; "
    f"protein union graph: {graphs['protein_edge_index'].shape[1]}"
)


#### Figure 3 — Spatial neighborhood structure (one component of the encoder graph)

Drawing every edge on all ~4,000 spots makes the local neighborhood structure
unreadable, so this figure splits overview and zoom for the **spatial 6-NN**
component only. Remember: each DGAT encoder also unions a molecular 10-NN graph.

- **Left:** all spots as nodes (no edges), with a red box marking the corner region.
- **Right:** undirected spatial 6-nearest-neighbor edges inside that corner.
- **Highlight:** the red spot is a boundary example; the orange spots are its spatial
  neighbors in this teaching view. The full encoder edge set is larger once molecular
  neighbors are unioned (saved below as `rna_edge_index` / `protein_edge_index`).


In [ ]:
fig, _ = plot_spatial_knn_graph(spots, edge_index, n_neighbors=6, zoom_corner="lower_left")
graph_path = paths.figures / "session01_spatial_knn_graph.png"
fig.savefig(graph_path, dpi=160, bbox_inches="tight")
plt.show()


#### Figure 4 — Normalized RNA and protein landscapes


In [ ]:
rna_features = rna.var(axis=0).nlargest(3).index
protein_features = protein.var(axis=0).nlargest(3).index
fig, axes = plt.subplots(2, 3, figsize=(13, 8))
for ax, feature in zip(axes[0], rna_features):
    plot_spatial_feature(spots, rna[feature], f"Normalized RNA: {feature}", cmap="magma", ax=ax)
for ax, feature in zip(axes[1], protein_features):
    plot_spatial_feature(spots, protein[feature], f"CLR protein: {feature}", cmap="viridis", ax=ax)
figure_path = paths.figures / "session01_normalized_spatial_features.png"
fig.savefig(figure_path, dpi=160, bbox_inches="tight")
plt.show()


#### Figure 5 — Exploratory RNA and protein embeddings

Colors are exploratory KMeans groups in PCA space (**not** curated cell-type labels).

- **Left:** RNA PCA; spots share one of four RNA cluster colors (`0`–`3`).
- **Middle:** protein PCA with its **own** four protein clusters (same palette, independent IDs).
- **Right:** the RNA cluster colors from the left panel mapped back onto tissue coordinates.

Use these panels only to check whether modality structure and spatial organization look coherent before moving to DGAT.


In [ ]:
def pca_clusters(matrix, n_clusters=4, n_pcs=20):
    n_pcs = min(n_pcs, matrix.shape[0] - 1, matrix.shape[1])
    pcs = PCA(n_components=n_pcs, random_state=7).fit_transform(matrix)
    clusters = KMeans(n_clusters=min(n_clusters, len(matrix)), n_init=20, random_state=7).fit_predict(pcs)
    return pcs[:, :2], clusters  # visualize PC1/PC2; cluster in higher-D PC space


def cluster_colors(labels, cmap_name="tab10"):
    """Map integer cluster IDs to stable discrete colors."""
    cmap = plt.get_cmap(cmap_name)
    return np.asarray([cmap(int(label) % 10) for label in labels])


rna_pca, rna_cluster = pca_clusters(rna)
protein_pca, protein_cluster = pca_clusters(protein)
fig, axes = plt.subplots(1, 3, figsize=(13, 3.8))
axes[0].scatter(rna_pca[:, 0], rna_pca[:, 1], c=cluster_colors(rna_cluster), s=20)
axes[0].set_title("RNA PCA (exploratory clusters)")
axes[0].set_xlabel("PC1")
axes[0].set_ylabel("PC2")

axes[1].scatter(protein_pca[:, 0], protein_pca[:, 1], c=cluster_colors(protein_cluster), s=20)
axes[1].set_title("Protein PCA (exploratory clusters)")
axes[1].set_xlabel("PC1")
axes[1].set_ylabel("PC2")

axes[2].scatter(spots["x"], spots["y"], c=cluster_colors(rna_cluster), s=20)
axes[2].set_title("RNA clusters mapped to tissue")
axes[2].set_xlabel("x")
axes[2].set_ylabel("y")
axes[2].set_aspect("equal")

embedding_path = paths.figures / "session01_modality_pca_and_spatial_clusters.png"
fig.tight_layout()
fig.savefig(embedding_path, dpi=160, bbox_inches="tight")
plt.show()


### Save the graph and checkpoint


In [ ]:
edge_table = pd.DataFrame({"source": spots.index[edge_index[0]], "target": spots.index[edge_index[1]]})
edge_path = paths.results / "session01_spatial_knn_edges.csv"
edge_table.to_csv(edge_path, index=False)
manifest = write_checkpoint(
    "1.3", [edge_path, graph_path, figure_path, embedding_path],
    summary={"nodes": len(spots), "directed_edges": edge_index.shape[1]}, start=paths.root,
)
print(f"Checkpoint written: {manifest}")


### Check

Trace one spot from its normalized feature vector to its node and six outgoing spatial edges. DGAT's
graph attention layers learn how much neighbor information to aggregate; the graph defines which
neighbors are available.
